# Audiobook Engine: Stage 2, repaired Chatterbox V3 pronunciation auditions

This notebook replaces the earlier Stage 2 runner. It pins the **upstream Chatterbox source commit** that supports the explicit V3 selector and omits the old alignment-stream analyser. It never monkey-patches `model.generate`, never uses a nonexistent `generation_config`, and checkpoints each successful WAV so an interrupted run can resume.

**Start with a fresh Colab runtime.** The old session may contain nested monkey-patches from previous attempts. Use **Runtime → Disconnect and delete runtime**, select a GPU runtime, open this notebook fresh from GitHub, and run from the top. This does not delete files in Google Drive.

## 1. Mount Drive and verify the approved manuscript checkpoint

This stage consumes the already-approved text and pronunciation sheet. It does not re-extract the PDF, edit the manuscript, or authorize book synthesis.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, shutil, inspect, hashlib, csv, traceback, gc
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

REPOSITORY = 'https://github.com/privacyfreak84/colabchatterLLMthingstuff.git'
PROJECT_DIR = Path('/content/colabchatterLLMthingstuff')
if (PROJECT_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin', 'main:refs/remotes/origin/main'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', '-B', 'main', 'origin/main'], check=True)
elif not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', REPOSITORY, str(PROJECT_DIR)], check=True)
else:
    raise RuntimeError(f'{PROJECT_DIR} exists but is not a Git repository.')

sys.path.insert(0, str(PROJECT_DIR))
OUTPUT_DIR = Path('/content/drive/MyDrive/Audiobook/Buddhist Yoga/reconnaissance')
required = [
    'main_manuscript_approved.txt', 'notes_manuscript_approved.txt',
    'manuscript_approval_record.json', 'pronunciation_audition_sheet.csv',
]
missing = [name for name in required if not (OUTPUT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing approved Stage 1 files: {missing}')

approval = json.loads((OUTPUT_DIR / 'manuscript_approval_record.json').read_text(encoding='utf-8'))
assert approval['accepted_cross_page_join_count'] == 48, approval
assert approval['synthesis_authorized'] is False, 'Full-book synthesis must remain locked.'
assert [x['candidate_id'] for x in approval['excluded_manual_review_candidates']] == ['J048']

CHATTERBOX_COMMIT = '5de7a54aa4e5e2baadb0182dde554908b48b85c2'
print('Approved manuscript confirmed: 48 joins accepted; J048 excluded; synthesis locked.')
print('Project revision:', subprocess.check_output(['git', '-C', str(PROJECT_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())

## 2. Install the correct Chatterbox source and verify its API

The PyPI `0.1.7` wheel does **not** expose the `t3_model` argument and still contains the old repetition/alignment analyser. This notebook installs a pinned upstream Git commit instead. The package is installed with the upstream source revision, then its inference dependencies are installed separately. The existing Colab Torch and torchaudio versions are recorded and must remain unchanged.

In [ ]:
import torch, torchaudio
TORCH_BEFORE = torch.__version__
TORCHAUDIO_BEFORE = torchaudio.__version__
print('Existing Torch:', TORCH_BEFORE, '| torchaudio:', TORCHAUDIO_BEFORE)
if sys.version_info < (3, 13) and int(np.__version__.split('.')[0]) >= 2:
    raise RuntimeError('This pinned Chatterbox dependency set expects NumPy <2 on Python <3.13. Run `%pip install -q "numpy<2"`, then Runtime > Restart session and rerun from the top.')

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'chatterbox-tts'], check=False)
source_spec = f'git+https://github.com/resemble-ai/chatterbox.git@{CHATTERBOX_COMMIT}'
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--no-deps', source_spec
], check=True)

dependency_specs = [
    'librosa==0.11.0', 's3tokenizer', 'transformers==5.2.0', 'diffusers==0.29.0',
    'git+https://github.com/resemble-ai/Perth.git@master', 'conformer==0.3.2',
    'safetensors==0.5.3', 'omegaconf', 'soundfile',
]
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', *dependency_specs
], check=True)

import librosa, soundfile as sf
from chatterbox.mtl_tts import ChatterboxMultilingualTTS, punc_norm
from chatterbox.models.t3 import T3

assert torch.__version__ == TORCH_BEFORE, f'Torch changed during install: {TORCH_BEFORE} -> {torch.__version__}'
assert torchaudio.__version__ == TORCHAUDIO_BEFORE, f'torchaudio changed during install: {TORCHAUDIO_BEFORE} -> {torchaudio.__version__}'
torch_base = torch.__version__.split('+')[0].split('.')[:2]
torchaudio_base = torchaudio.__version__.split('+')[0].split('.')[:2]
assert torch_base == torchaudio_base, f'Torch/torchaudio mismatch: {TORCH_BEFORE} vs {TORCHAUDIO_BEFORE}'
pretrained_params = inspect.signature(ChatterboxMultilingualTTS.from_pretrained).parameters
local_params = inspect.signature(ChatterboxMultilingualTTS.from_local).parameters
assert 't3_model' in pretrained_params and 't3_model' in local_params, (
    'Wrong Chatterbox package is installed. The V3 selector is missing.'
)
t3_inference_source = inspect.getsource(T3.inference)
assert 'alignment_stream_analyzer' not in t3_inference_source, (
    'The legacy alignment analyser is still present. Refusing to run the old repetition guard.'
)
print('Chatterbox source commit pinned:', CHATTERBOX_COMMIT)
print('V3 selector available:', 't3_model' in pretrained_params)
print('Legacy alignment analyser absent from T3 inference: confirmed')
print('Torch and torchaudio preserved.')

## 3. Download or reuse a clean, verified V3 snapshot

The previous cache was placed inside Google Drive's Hugging Face cache tree, which uses cache metadata and blob links. This notebook does **not** reuse or purge that global cache. It downloads into a new plain local folder, verifies the actual files, and copies the finished snapshot into a dedicated Drive folder as ordinary files. Only this notebook's own `chatterbox-v3-<revision>` folder is eligible for replacement.

In [ ]:
# Keep Hugging Face's transient metadata/cache on the Colab VM, not in Drive.
HF_HOME = Path('/content/audiobook-hf-home')
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_HOME)
os.environ['HF_HUB_CACHE'] = str(HF_HOME / 'hub')

from huggingface_hub import HfApi, snapshot_download
MODEL_REPO = 'ResembleAI/chatterbox'
info = HfApi().model_info(MODEL_REPO, revision='main', files_metadata=True)
MODEL_REVISION = info.sha
MODEL_FILES = [
    've.pt', 's3gen.pt', 't3_mtl23ls_v3.safetensors',
    'grapheme_mtl_merged_expanded_v1.json', 'conds.pt', 'Cangjie5_TC.json',
]
remote_sizes = {
    item.rfilename: getattr(item, 'size', None)
    for item in info.siblings
    if item.rfilename in MODEL_FILES
}
minimum_sizes = {
    've.pt': 5_000_000,
    's3gen.pt': 1_000_000_000,
    't3_mtl23ls_v3.safetensors': 2_000_000_000,
    'grapheme_mtl_merged_expanded_v1.json': 60_000,
    'conds.pt': 50_000,
    'Cangjie5_TC.json': 1_000_000,
}

DRIVE_MODEL_DIR = Path('/content/drive/MyDrive/Audiobook/model-snapshots') / f'chatterbox-v3-{MODEL_REVISION[:12]}'
LOCAL_MODEL_DIR = Path('/content/chatterbox-v3-model')
def check_snapshot(folder):
    problems = []
    for name in MODEL_FILES:
        path = folder / name
        if not path.is_file():
            problems.append(f'{name}: missing')
            continue
        actual = path.stat().st_size
        expected = remote_sizes.get(name)
        if expected and actual != expected:
            problems.append(f'{name}: {actual:,} bytes, Hub reports {expected:,}')
        elif not expected and actual < minimum_sizes[name]:
            problems.append(f'{name}: suspiciously small ({actual:,} bytes)')
    return problems

if LOCAL_MODEL_DIR.exists():
    shutil.rmtree(LOCAL_MODEL_DIR)
LOCAL_MODEL_DIR.mkdir(parents=True)
if not check_snapshot(DRIVE_MODEL_DIR):
    print('Using the verified plain-file V3 snapshot already in Drive.')
    for name in MODEL_FILES:
        shutil.copy2(DRIVE_MODEL_DIR / name, LOCAL_MODEL_DIR / name)
else:
    if DRIVE_MODEL_DIR.exists():
        print('The dedicated V3 snapshot folder is incomplete or mismatched; replacing only that folder.')
        shutil.rmtree(DRIVE_MODEL_DIR)
    print('Downloading the selected V3 files to local Colab storage. This is not the full 13.9 GB repository.')
    snapshot_download(
        repo_id=MODEL_REPO,
        repo_type='model',
        revision=MODEL_REVISION,
        allow_patterns=MODEL_FILES,
        local_dir=str(LOCAL_MODEL_DIR),
        force_download=True,
    )
    problems = check_snapshot(LOCAL_MODEL_DIR)
    if problems:
        raise RuntimeError('Hub snapshot verification failed:\n' + '\n'.join(problems))
    try:
        DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
        for name in MODEL_FILES:
            shutil.copy2(LOCAL_MODEL_DIR / name, DRIVE_MODEL_DIR / name)
        (DRIVE_MODEL_DIR / 'snapshot_manifest.json').write_text(json.dumps({
            'repo': MODEL_REPO, 'revision': MODEL_REVISION, 'files': {
                name: {'size_bytes': (DRIVE_MODEL_DIR / name).stat().st_size}
                for name in MODEL_FILES
            }
        }, indent=2), encoding='utf-8')
        print('Verified model files persisted in Drive for future sessions.')
    except Exception as exc:
        print('Drive persistence could not complete:', type(exc).__name__, str(exc))
        print('The local verified snapshot remains available for this runtime; a future runtime may need to download it again.')

problems = check_snapshot(LOCAL_MODEL_DIR)
if problems:
    raise RuntimeError('Local model snapshot is incomplete:\n' + '\n'.join(problems))
print('V3 snapshot:', MODEL_REVISION)
for name in MODEL_FILES:
    print(f'{name}: {(LOCAL_MODEL_DIR / name).stat().st_size / 1024**2:.1f} MiB')

## 4. Preflight the checkpoint files and load V3

No `torch.load` monkey-patch is used. The `.pt` files are read in PyTorch's safe `weights_only=True` mode before model construction. If a checkpoint is corrupt, only that file is force-downloaded once. The rest of the Hub cache and all unrelated Drive files are left alone.

In [ ]:
import torch
from safetensors import safe_open

def safe_checkpoint_preflight(filename, force_refresh=False):
    path = LOCAL_MODEL_DIR / filename
    try:
        state = torch.load(path, map_location='cpu', weights_only=True, mmap=True)
        if not isinstance(state, dict) or not state:
            raise RuntimeError(f'{filename} did not contain a non-empty state dictionary.')
        print(f'{filename}: safe PyTorch checkpoint read, {len(state)} top-level entries')
        del state
        gc.collect()
    except Exception as exc:
        if force_refresh:
            raise RuntimeError(f'{filename} still fails after a fresh Hub download: {type(exc).__name__}: {exc}') from exc
        print(f'{filename} failed preflight ({type(exc).__name__}: {exc}). Refreshing this file only.')
        snapshot_download(
            repo_id=MODEL_REPO, repo_type='model', revision=MODEL_REVISION,
            allow_patterns=[filename], local_dir=str(LOCAL_MODEL_DIR), force_download=True,
        )
        safe_checkpoint_preflight(filename, force_refresh=True)
        shutil.copy2(path, DRIVE_MODEL_DIR / filename)

safe_checkpoint_preflight('ve.pt')
safe_checkpoint_preflight('s3gen.pt')
with safe_open(str(LOCAL_MODEL_DIR / 't3_mtl23ls_v3.safetensors'), framework='pt', device='cpu') as handle:
    assert len(handle.keys()) > 0, 'V3 T3 safetensors file is empty.'
    print('t3_mtl23ls_v3.safetensors: safe header read,', len(handle.keys()), 'tensors')

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Select a GPU runtime in Colab before loading V3.')
gpu = torch.cuda.get_device_properties(0)
print('GPU:', gpu.name, '| VRAM GiB:', round(gpu.total_memory / 1024**3, 1))
print('Loading V3 from the verified local snapshot...')
model = ChatterboxMultilingualTTS.from_local(
    str(LOCAL_MODEL_DIR), device='cuda', t3_model='v3'
)
assert int(model.sr) == 24000, f'Unexpected Chatterbox sample rate: {model.sr}'
supported = ChatterboxMultilingualTTS.get_supported_languages()
assert {'en', 'hi'}.issubset(supported), supported
print('Loaded Multilingual V3:', supported['en'], '+', supported['hi'], '| sample rate:', model.sr)

## 5. Prepare aligned English and Hindi reference clips

The reference-length warning is caused by a frame-count mismatch between the 24 kHz mel features and 16 kHz speech tokens. The upstream S3Gen code handles it by trimming the token sequence, but this cell first pads the demo clips to an exact multiple of 40 ms to avoid the common mismatch. If the warning still appears, it is not the 2x repetition warning and is not itself a fatal error.

In [ ]:
REFERENCE_DIR = OUTPUT_DIR / 'pronunciation_auditions' / 'references'
REFERENCE_DIR.mkdir(parents=True, exist_ok=True)
EN_SOURCE = REFERENCE_DIR / 'official_demo_en_f1.flac'
HI_SOURCE = REFERENCE_DIR / 'official_demo_hi_f1.flac'
EN_REF = REFERENCE_DIR / 'official_demo_en_f1_padded.wav'
HI_REF = REFERENCE_DIR / 'official_demo_hi_f1_padded.wav'
urls = {
    EN_SOURCE: 'https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/en_f1.flac',
    HI_SOURCE: 'https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/hi_f1.flac',
}
for path, url in urls.items():
    if not path.is_file() or path.stat().st_size < 1024:
        import urllib.request
        urllib.request.urlretrieve(url, path)

def make_aligned_reference(source_path, output_path):
    audio, sr = sf.read(source_path, dtype='float32', always_2d=False)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    if audio.ndim != 1 or audio.size == 0 or not np.isfinite(audio).all():
        raise ValueError(f'Invalid reference audio: {source_path}')
    if sr != 24000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=24000)
    audio = audio[:24000 * 10]
    if audio.size < 24000:
        raise ValueError(f'Reference clip is shorter than one second: {source_path}')
    pad = (-audio.size) % 960  # 40 ms at 24 kHz; 16 kHz counterpart is 640 samples.
    if pad:
        audio = np.pad(audio, (0, pad))
    sf.write(output_path, audio, 24000, subtype='PCM_16')
    print(output_path.name, '| duration:', round(audio.size / 24000, 3), 's', '| samples mod 960:', audio.size % 960)

make_aligned_reference(EN_SOURCE, EN_REF)
make_aligned_reference(HI_SOURCE, HI_REF)

## 6. Two-sample smoke test before the 36 audition tasks

First generate exactly one English and one Hindi sample. Listen to these before launching the full pronunciation set. This gives us a small, useful diagnostic if the runtime or reference conditioning is still unhealthy.

In [ ]:
from IPython.display import Audio, Markdown, display
SMOKE_PARAMETERS = {
    'exaggeration': 0.5, 'cfg_weight': 0.5, 'temperature': 0.7,
    'repetition_penalty': 1.2, 'min_p': 0.05, 'top_p': 0.95,
}
SMOKE_DIR = OUTPUT_DIR / 'pronunciation_auditions' / 'v3-smoke'
SMOKE_DIR.mkdir(parents=True, exist_ok=True)

model.prepare_conditionals(str(EN_REF), exaggeration=SMOKE_PARAMETERS['exaggeration'])
en_smoke = model.generate(
    'Maitreya spoke to the Buddha.', language_id='en', **SMOKE_PARAMETERS
)
en_smoke_path = SMOKE_DIR / 'smoke_english.wav'
sf.write(en_smoke_path, en_smoke.detach().float().cpu().numpy().squeeze(), int(model.sr), subtype='PCM_16')

model.prepare_conditionals(str(HI_REF), exaggeration=SMOKE_PARAMETERS['exaggeration'])
hi_smoke = model.generate(
    'मैत्रेय ने कहा।', language_id='hi', **SMOKE_PARAMETERS
)
hi_smoke_path = SMOKE_DIR / 'smoke_hindi.wav'
sf.write(hi_smoke_path, hi_smoke.detach().float().cpu().numpy().squeeze(), int(model.sr), subtype='PCM_16')

display(Markdown('**English V3 smoke test**'))
display(Audio(filename=str(en_smoke_path), autoplay=False))
display(Markdown('**Hindi V3 smoke test**'))
display(Audio(filename=str(hi_smoke_path), autoplay=False))
print('Both smoke samples generated. No generate() monkey-patch has been applied.')

### Listen before continuing

Running every cell from the top must **stop here**. Listen to both smoke-test clips first. Only after they sound usable should you opt in to the 36 pronunciation tasks in the next cell.

In [ ]:
RUN_FULL_AUDITIONS = False  # Change to True only after listening to both smoke-test clips.
print('Full audition generation enabled:' if RUN_FULL_AUDITIONS else 'Smoke-test only. Full audition generation is still OFF.')


## 7. Generate the pronunciation audition set

The helper now passes supported sampling parameters directly to `model.generate()`. It does not accept unsupported arguments such as `temperature` at the wrapper level in older versions, because this pinned helper explicitly exposes them as a `generation_parameters` dictionary. Each sample is checkpointed immediately. If generation fails or is interrupted, the first failure traceback is saved and the run stops rather than producing 36 identical failure rows.

In [ ]:
AUDITION_DIR = OUTPUT_DIR / 'pronunciation_auditions' / 'v3-upstream-auditions'
AUDITION_DIR.mkdir(parents=True, exist_ok=True)

if not RUN_FULL_AUDITIONS:
    print('Stopped after the two smoke tests. Listen first, then set RUN_FULL_AUDITIONS = True and rerun this cell.')
else:
    from tools.pronunciation_audition import generate_audition_set

    audition_summary = generate_audition_set(
        model=model,
        audition_sheet=OUTPUT_DIR / 'pronunciation_audition_sheet.csv',
        output_dir=AUDITION_DIR,
        english_reference_audio=EN_REF,
        hindi_reference_audio=HI_REF,
        punc_norm=punc_norm,
        model_revision=MODEL_REVISION,
        code_revision=CHATTERBOX_COMMIT,
        seed_base=20261030,
        generation_parameters=SMOKE_PARAMETERS,
    )
    print(json.dumps(audition_summary, ensure_ascii=False, indent=2))
    if audition_summary['generated_count'] == 0:
        raise RuntimeError('No audition audio was generated. See the task CSV and traceback files.')

## 8. Listen to the completed pronunciation comparisons

The Hindi reference is a Hindi voice, while English/IAST use an English reference. Use the Hindi samples to judge the phonetic rendering, not to compare narrator identity or voice quality across languages.

In [ ]:
decision_csv = AUDITION_DIR / 'pronunciation_audition_decisions.csv'
if not decision_csv.is_file():
    print('The full audition set has not been generated. Listen to the smoke tests, opt in, then run the generation cell first.')
    decision_rows = []
else:
    with decision_csv.open('r', encoding='utf-8-sig', newline='') as handle:
        decision_rows = list(csv.DictReader(handle))
audio_columns = [
    ('Source spelling, English V3', 'source_english_audio'),
    ('IAST spelling, English V3', 'iast_english_audio'),
    ('Devanagari, Hindi V3', 'devanagari_hindi_audio'),
]
for row in decision_rows:
    display(Markdown(f"### {row['audition_id']}: {row['source_term']} ({row['occurrences']} occurrences)"))
    for label, field in audio_columns:
        filename = row.get(field, '')
        if filename:
            display(Markdown(f'**{label}**'))
            display(Audio(filename=str(AUDITION_DIR / filename), autoplay=False))
        elif field == 'iast_english_audio':
            display(Markdown('_Same spelling as source; duplicate audio was skipped._'))
        else:
            display(Markdown('_No WAV recorded. Inspect pronunciation_audition_tasks.csv._'))

## 9. Package the checkpointed audition evidence

The bundle includes successful WAVs, the exact model and code revisions, tokenizer preflight results, per-task seeds and hashes, failure tracebacks (if any), and the blank decision sheet. It does not contain or authorize full-book narration.

In [ ]:
import zipfile, base64
if not (AUDITION_DIR / 'pronunciation_audition_summary.json').is_file():
    print('Bundle packaging skipped: the smoke test is ready, but full auditions are still disabled.')
else:
    zip_path = OUTPUT_DIR / 'pronunciation_auditions_bundle.zip'
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(AUDITION_DIR.rglob('*')):
            if path.is_file():
                archive.write(path, path.relative_to(AUDITION_DIR))
    encoded_path = OUTPUT_DIR / 'pronunciation_auditions_bundle.txt'
    encoded_path.write_text(base64.b64encode(zip_path.read_bytes()).decode('ascii') + '\n', encoding='ascii')
    print(json.dumps({
        'zip_path': str(zip_path), 'zip_bytes': zip_path.stat().st_size,
        'zip_sha256': hashlib.sha256(zip_path.read_bytes()).hexdigest(),
        'transfer_text_file': str(encoded_path), 'base64_chars': encoded_path.stat().st_size,
        'full_book_synthesis_authorized': False,
    }, ensure_ascii=False, indent=2))
    print('Upload the transfer text file to the repository as nextrecon2.txt.')

## Stop before full-book synthesis

Do not start the audiobook. We must inspect the audition bundle, listen to the variants, record pronunciation decisions, and approve the final reversible narration substitutions first. If the smoke test or a task fails, use its saved traceback; do not patch `model.generate`, change `torch.load(weights_only=...)`, or wipe the global Hugging Face cache.

## 10. Optional English pronunciation refinement probes

The first audition batch established that Hindi-mode Devanagari samples are useful, but raw English spelling and IAST are not reliable pronunciation instructions for Sanskrit terms. This optional, separate probe tests two alternatives on four representative terms:

1. **English phonetic respelling:** English V3 reads a provisional syllable-by-syllable cue.
2. **Hindi-script reading with the English reference voice:** Hindi V3 reads the Devanagari form while using the same English reference clip. The cross-language CFG weight is set to 0.0, following the upstream guidance for reducing accent transfer.

These spellings are provisional approximations for listening, not approved dictionary entries or a replacement for the source manuscript. The test does not authorize full-book synthesis. Leave the gate off until you explicitly choose to run it.

In [ ]:
# Separate opt-in: these eight short probes are not part of the original 36-task batch.
RUN_ENGLISH_REFINEMENT_PROBES = False  # Set True only when ready to generate the new comparison clips.

if not RUN_ENGLISH_REFINEMENT_PROBES:
    print("English refinement probes are OFF. Set RUN_ENGLISH_REFINEMENT_PROBES = True and rerun this cell.")
else:
    import hashlib
    import json
    import zipfile
    import base64
    import torch
    import numpy as np
    import soundfile as sf
    from IPython.display import Audio, Markdown, display
    from tools.pronunciation_audition import DEVANAGARI_FORMS

    # A small diagnostic set: complex names plus a term with the Sanskrit jñ consonant cluster.
    # Cues are deliberately provisional; judge the audio rather than treating these strings as approved.
    PHONETIC_FORMS = {
        "Avalokiteshvara": "uh-vuh-loh-kee-tay-shva-ruh",
        "Maitreya": "my-tray-yuh",
        "Manjushri": "mun-joo-shree",
        "Vijnanavada": "vig-nyaa-nuh-vaa-duh",
    }
    REFINEMENT_DIR = OUTPUT_DIR / "pronunciation_auditions" / "v3-english-refinement-probes"
    REFINEMENT_DIR.mkdir(parents=True, exist_ok=True)
    REFINEMENT_MANIFEST = REFINEMENT_DIR / "english_refinement_manifest.json"

    with (OUTPUT_DIR / "pronunciation_audition_sheet.csv").open("r", encoding="utf-8-sig", newline="") as handle:
        sheet_rows = list(csv.DictReader(handle))
    sheet_by_term = {row["source_term"]: row for row in sheet_rows}
    if set(PHONETIC_FORMS) - set(sheet_by_term):
        raise RuntimeError("The refinement terms do not match the approved pronunciation sheet.")

    specifications = []
    for index, (term, cue) in enumerate(PHONETIC_FORMS.items(), start=1):
        devanagari = DEVANAGARI_FORMS[term]
        specifications.extend([
            {
                "task_id": f"R{index:03d}_EN_PHONETIC",
                "term": term,
                "variant": "english_phonetic_respelling",
                "language_id": "en",
                "text": f"The Sanskrit name is {cue}.",
                "reference_audio": EN_REF.name,
                "generation_parameters": dict(SMOKE_PARAMETERS),
                "seed": 20261031 + index * 10,
            },
            {
                "task_id": f"R{index:03d}_HI_EN_REFERENCE",
                "term": term,
                "variant": "hindi_devanagari_english_reference_voice",
                "language_id": "hi",
                "text": f"इस शब्द का उच्चारण {devanagari} है।",
                "reference_audio": EN_REF.name,
                "generation_parameters": {**SMOKE_PARAMETERS, "cfg_weight": 0.0},
                "seed": 20261031 + index * 10 + 1,
            },
        ])

    def _probe_sha256(path):
        digest = hashlib.sha256()
        with path.open("rb") as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(block)
        return digest.hexdigest()

    if REFINEMENT_MANIFEST.is_file():
        old = json.loads(REFINEMENT_MANIFEST.read_text(encoding="utf-8"))
        if old.get("model_revision") != MODEL_REVISION or old.get("chatterbox_source_commit") != CHATTERBOX_COMMIT:
            raise RuntimeError("This probe folder belongs to a different model/code revision. Preserve it and choose a fresh output folder.")
        if old.get("reference_sha256") != _probe_sha256(EN_REF):
            raise RuntimeError("The English reference audio changed. Preserve this probe folder and choose a fresh output folder.")
        old_by_id = {item["task_id"]: item for item in old.get("tasks", [])}
        for spec in specifications:
            prior = old_by_id.get(spec["task_id"])
            if prior and any(prior.get(key) != spec.get(key) for key in (
                "term", "variant", "language_id", "text", "reference_audio",
                "generation_parameters", "seed",
            )):
                raise RuntimeError(f"Probe settings changed for {spec['task_id']}. Preserve old evidence and choose a fresh output folder.")
    else:
        old_by_id = {}

    probe_manifest = {
        "schema_version": "1.0.0",
        "model": "ResembleAI/chatterbox, Multilingual V3",
        "model_revision": MODEL_REVISION,
        "chatterbox_source_commit": CHATTERBOX_COMMIT,
        "english_reference_audio": EN_REF.name,
        "reference_sha256": _probe_sha256(EN_REF),
        "english_phonetic_forms_are_provisional": True,
        "full_book_synthesis_authorized": False,
        "tasks": [],
    }
    probe_by_id = {}
    for spec in specifications:
        prior = old_by_id.get(spec["task_id"], {})
        task = {**spec, **{k: v for k, v in prior.items() if k not in spec}}
        wav_path = REFINEMENT_DIR / f"{task['task_id']}.wav"
        if prior.get("status") == "generated" and wav_path.is_file() and prior.get("sha256") == _probe_sha256(wav_path):
            task["status"] = "generated"
            task["output_file"] = wav_path.name
        else:
            task["status"] = "pending"
            task["output_file"] = ""
            task["sha256"] = ""
        probe_by_id[task["task_id"]] = task

    def save_probe_manifest():
        probe_manifest["tasks"] = list(probe_by_id.values())
        temp = REFINEMENT_MANIFEST.with_suffix(".json.tmp")
        temp.write_text(json.dumps(probe_manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        temp.replace(REFINEMENT_MANIFEST)

    # Both routes use the same English reference recording. Only the spoken language/script and CFG setting differ.
    model.prepare_conditionals(str(EN_REF), exaggeration=SMOKE_PARAMETERS["exaggeration"])
    for spec in specifications:
        task = probe_by_id[spec["task_id"]]
        wav_path = REFINEMENT_DIR / f"{task['task_id']}.wav"
        if task["status"] == "generated":
            print("Reusing verified:", task["task_id"])
            continue

        seed = int(task["seed"])
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)

        task["status"] = "running"
        save_probe_manifest()
        try:
            wav = model.generate(
                task["text"],
                language_id=task["language_id"],
                **task["generation_parameters"],
            )
            waveform = wav.detach().float().cpu().numpy().squeeze()
            if waveform.ndim != 1 or waveform.size == 0 or not np.isfinite(waveform).all():
                raise ValueError(f"Invalid waveform for {task['task_id']}: {getattr(wav, 'shape', None)}")
            temp_wav = REFINEMENT_DIR / f"{task['task_id']}.tmp.wav"
            sf.write(str(temp_wav), waveform, int(model.sr), format="WAV", subtype="PCM_16")
            temp_wav.replace(wav_path)
            task["output_file"] = wav_path.name
            task["duration_seconds"] = round(waveform.size / int(model.sr), 3)
            task["sha256"] = _probe_sha256(wav_path)
            task["status"] = "generated"
            task["error"] = ""
            save_probe_manifest()
            print(f"{task['task_id']}: {task['duration_seconds']} s")
        except BaseException as exc:
            task["status"] = "interrupted" if isinstance(exc, KeyboardInterrupt) else "generation_failed"
            task["error"] = f"{type(exc).__name__}: {exc}"
            task["traceback"] = traceback.format_exc()
            save_probe_manifest()
            raise

    save_probe_manifest()
    display(Markdown("### English refinement comparison clips"))
    for index, term in enumerate(PHONETIC_FORMS, start=1):
        display(Markdown(f"**{term}**"))
        for task_id, label in [
            (f"R{index:03d}_EN_PHONETIC", "English mode, phonetic respelling"),
            (f"R{index:03d}_HI_EN_REFERENCE", "Hindi mode, same English reference voice"),
        ]:
            task = probe_by_id[task_id]
            display(Markdown(f"*{label}*"))
            display(Audio(filename=str(REFINEMENT_DIR / task["output_file"]), autoplay=False))

    zip_path = OUTPUT_DIR / "pronunciation_refinement_probes_bundle.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(REFINEMENT_DIR.rglob("*")):
            if path.is_file():
                archive.write(path, path.relative_to(REFINEMENT_DIR))
    transfer_path = OUTPUT_DIR / "pronunciation_refinement_probes_bundle.txt"
    transfer_path.write_text(base64.b64encode(zip_path.read_bytes()).decode("ascii") + "\n", encoding="ascii")
    print("Probe bundle:", transfer_path)
    print("Upload it to the repository as nextrecon3.txt after listening.")


## 11. Hindi-to-English voice bridge experiments

This pass tests two Chatterbox-native routes while keeping Hindi pronunciation generation and English narrator rendering separate:

1. **Split conditioning:** prepare T3 from the Hindi reference and independently prepare S3Gen from the English target-voice reference. Generate Devanagari Hindi with language_id="hi" while S3Gen renders through the English reference.
2. **ChatterboxVC:** generate Hindi source speech using matching Hindi T3/S3Gen conditioning, then convert the resulting audio with the same V3 S3Gen model and the English target-voice reference.

Four representative Sanskrit terms receive six clips each (24 total): Hindi source baseline and pace preset; split-conditioning baseline and pace preset; and voice-converted baseline and pace-preset sources.

The pace preset (exaggeration=0.7, cfg_weight=0.3) is an experiment based on upstream Chatterbox guidance. It is not assumed to be better. These auditions use the public English/Hindi demo references, not the final audiobook narrator. They test feasibility and must be followed by a second audition using the actual narrator reference if a route passes.

This section reuses the loaded V3 model and prepared reference files. It does not rerun the earlier 36 tasks or the eight rejected English probes. It does not edit the approved manuscript or authorize full-book synthesis.

### How to run

Leave the gate off until ready. Set RUN_HINDI_ENGLISH_BRIDGE_EXPERIMENTS = True and execute this cell. It checkpoints every task, reuses hash-verified WAVs on reruns, writes a manifest, plays all comparisons, and creates hindi_english_bridge_bundle.txt. Upload that transfer file to the repository root as nextrecon4.txt after listening.


In [ ]:
# Explicit opt-in: 24 short diagnostic clips, not full-book synthesis.
RUN_HINDI_ENGLISH_BRIDGE_EXPERIMENTS = False

if not RUN_HINDI_ENGLISH_BRIDGE_EXPERIMENTS:
    print("Hindi-to-English bridge experiments are OFF. Set the gate to True and rerun this cell when ready.")
else:
    import base64
    import hashlib
    import json
    import zipfile
    from pathlib import Path
    from IPython.display import Audio, Markdown, display
    from tools.chatterbox_voice_bridge import run_hindi_english_bridge_auditions

    BRIDGE_DIR = OUTPUT_DIR / "pronunciation_auditions" / "v3-hindi-english-bridges"
    BRIDGE_DIR.mkdir(parents=True, exist_ok=True)

    bridge_summary = run_hindi_english_bridge_auditions(
        model=model,
        english_reference_audio=EN_REF,
        hindi_reference_audio=HI_REF,
        output_dir=BRIDGE_DIR,
        devanagari_forms={
            "Avalokiteshvara": "अवलोकितेश्वर",
            "Maitreya": "मैत्रेय",
            "Manjushri": "मञ्जुश्री",
            "Vijnanavada": "विज्ञानवाद",
        },
        model_revision=MODEL_REVISION,
        code_revision=CHATTERBOX_COMMIT,
        base_generation_parameters=SMOKE_PARAMETERS,
    )

    print(json.dumps({
        "task_count": bridge_summary["task_count"],
        "generated_count": bridge_summary["generated_count"],
        "failed_count": bridge_summary["failed_count"],
        "manifest_path": bridge_summary["manifest_path"],
        "full_book_synthesis_authorized": bridge_summary["full_book_synthesis_authorized"],
    }, ensure_ascii=False, indent=2))

    # Group players by Sanskrit term so the six routes can be compared in one sitting.
    for term in ["Avalokiteshvara", "Maitreya", "Manjushri", "Vijnanavada"]:
        display(Markdown(f"### {term}"))
        term_tasks = [task for task in bridge_summary["tasks"] if task["term"] == term]
        for task in term_tasks:
            display(Markdown(
                f"**{task['variant']}** · {task.get('duration_seconds', '?')} s"
                f" · peak {task.get('peak', '?')} · RMS {task.get('rms', '?')}"
            ))
            display(Audio(filename=str(BRIDGE_DIR / task["output_file"]), autoplay=False))

    # Keep the transfer bundle focused: manifest plus generated WAVs only.
    bridge_zip = OUTPUT_DIR / "hindi_english_bridge_bundle.zip"
    with zipfile.ZipFile(bridge_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        manifest_path = Path(bridge_summary["manifest_path"])
        archive.write(manifest_path, manifest_path.name)
        for wav_path in sorted(BRIDGE_DIR.glob("*.wav")):
            archive.write(wav_path, wav_path.name)

    transfer_path = OUTPUT_DIR / "hindi_english_bridge_bundle.txt"
    transfer_path.write_text(
        base64.b64encode(bridge_zip.read_bytes()).decode("ascii") + "\n",
        encoding="ascii",
    )
    print("Bundle:", transfer_path)
    print("Upload this file to the repository root as nextrecon4.txt after listening.")
    print("Reminder: generated audio is not automatically quality-approved; listen for pronunciation, pace, voice match, and artifacts.")
